In [ ]:
import os
import pickle
import numpy as np
import pandas as pd
from tqdm import tqdm
import random

import torch
import torch.nn as nn
from scipy.sparse import csr_matrix
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler

print("LOAD ARTIFACTS")

df_final = pd.read_csv('../data/df_final_prepared.csv')

# EASE
ease_dir = "../data/ease_final"

B = np.load(f"{ease_dir}/B.npy")

with open(f"{ease_dir}/product_to_idx.pkl", "rb") as f:
    product_to_idx = pickle.load(f)

with open(f"{ease_dir}/idx_to_product.pkl", "rb") as f:
    idx_to_product = pickle.load(f)

# VAE
vae_dir = "../data/vae_final"
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

with open(f"{vae_dir}/item_to_idx.pkl", "rb") as f:
    item_to_idx_vae = pickle.load(f)

with open(f"{vae_dir}/idx_to_item.pkl", "rb") as f:
    idx_to_item_vae = pickle.load(f)

with open(f"{vae_dir}/lvl2_to_idx.pkl", "rb") as f:
    lvl2_to_idx = pickle.load(f)

with open(f"{vae_dir}/lvl1_to_idx.pkl", "rb") as f:
    lvl1_to_idx = pickle.load(f)

with open(f"{vae_dir}/lvl0_to_idx.pkl", "rb") as f:
    lvl0_to_idx = pickle.load(f)

print("B:", B.shape)
print("EASE items:", len(product_to_idx))
print("VAE items:", len(item_to_idx_vae))

LOAD ARTIFACTS
B: (27212, 27212)
EASE items: 27212
VAE items: 27212


In [ ]:
n_items = item_to_idx_vae.shape[1]
n_lvl2 = lvl2_to_idx.shape[1]
n_lvl1 = lvl1_to_idx.shape[1]
n_lvl0 = lvl0_to_idx.shape[1]

In [5]:
# restore vae
class VAE(nn.Module):
    def __init__(self):
        super().__init__()
        
        self.item_enc = nn.Sequential(
            nn.Linear(len(item_to_idx_vae), 512),
            nn.ReLU(),
            nn.Linear(512, 256)
        )
        self.lvl2_enc = nn.Sequential(
            nn.Linear(len(lvl2_to_idx), 128),
            nn.ReLU(),
            nn.Linear(128, 64)
        )
        self.lvl1_enc = nn.Sequential(
            nn.Linear(len(lvl1_to_idx), 64),
            nn.ReLU(),
            nn.Linear(64, 32)
        )
        self.lvl0_enc = nn.Sequential(
            nn.Linear(len(lvl0_to_idx), 32),
            nn.ReLU(),
            nn.Linear(32, 16)
        )
        
        self.fc_mu = nn.Linear(256 + 64 + 32 + 16, 64)
        self.fc_logvar = nn.Linear(256 + 64 + 32 + 16, 64)
        
        self.item_dec = nn.Linear(64, len(item_to_idx_vae))
        self.lvl2_dec = nn.Linear(64, len(lvl2_to_idx))
        self.lvl1_dec = nn.Linear(64, len(lvl1_to_idx))
        self.lvl0_dec = nn.Linear(64, len(lvl0_to_idx))

    def encode(self, x):
        h = torch.cat([
            self.item_enc(x['item']),
            self.lvl2_enc(x['lvl2']),
            self.lvl1_enc(x['lvl1']),
            self.lvl0_enc(x['lvl0'])
        ], dim=1)
        return self.fc_mu(h), self.fc_logvar(h)

    def reparam(self, mu, logvar, sample=True):
        if not sample:
            return mu
        std = torch.exp(0.5 * logvar)
        eps = torch.randn_like(std)
        return mu + eps * std

    def decode(self, z):
        return {
            'item': self.item_dec(z),
            'lvl2': self.lvl2_dec(z),
            'lvl1': self.lvl1_dec(z),
            'lvl0': self.lvl0_dec(z),
        }

    def forward(self, x, sample=True):
        mu, logvar = self.encode(x)
        z = self.reparam(mu, logvar, sample=sample)
        recon = self.decode(z)
        return recon, mu, logvar

model = VAE().to(device)
model.load_state_dict(torch.load(f"{vae_dir}/vae_model.pt", map_location=device))
model.eval()

/tmp/ipykernel_1250229/2829579452.py:66: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  model.load_state_dict(torch.load(f"{vae_dir}/vae_model.pt", map_location=device))


VAE(
  (item_enc): Sequential(
    (0): Linear(in_features=27212, out_features=512, bias=True)
    (1): ReLU()
    (2): Linear(in_features=512, out_features=256, bias=True)
  )
  (lvl2_enc): Sequential(
    (0): Linear(in_features=631, out_features=128, bias=True)
    (1): ReLU()
    (2): Linear(in_features=128, out_features=64, bias=True)
  )
  (lvl1_enc): Sequential(
    (0): Linear(in_features=185, out_features=64, bias=True)
    (1): ReLU()
    (2): Linear(in_features=64, out_features=32, bias=True)
  )
  (lvl0_enc): Sequential(
    (0): Linear(in_features=36, out_features=32, bias=True)
    (1): ReLU()
    (2): Linear(in_features=32, out_features=16, bias=True)
  )
  (fc_mu): Linear(in_features=368, out_features=64, bias=True)
  (fc_logvar): Linear(in_features=368, out_features=64, bias=True)
  (item_dec): Linear(in_features=64, out_features=27212, bias=True)
  (lvl2_dec): Linear(in_features=64, out_features=631, bias=True)
  (lvl1_dec): Linear(in_features=64, out_features=185, bi

In [19]:
# X_train / X_test

baskets = df_final.groupby('cheque_id')['product_id'].apply(list).reset_index()

rows, cols = [], []

for i, row in baskets.iterrows():
    for pid in row['product_id']:
        if pid in product_to_idx:
            rows.append(i)
            cols.append(product_to_idx[pid])

data = np.ones(len(rows), dtype=np.float32)
X = csr_matrix((data, (rows, cols)), shape=(len(baskets), len(product_to_idx)))

train_idx, test_idx = train_test_split(
    np.arange(X.shape[0]),
    test_size=0.2,
    random_state=42
)

X_train = X[train_idx]
X_test  = X[test_idx]

print("X_train:", X_train.shape)
print("X_test :", X_test.shape)

X_train: (39469, 27212)
X_test : (9868, 27212)


In [20]:
# space mapping

meta = df_final[['product_id', 'name_clean',
                 'art_grp_lvl_2_name',
                 'art_grp_lvl_1_name',
                 'art_grp_lvl_0_name']].drop_duplicates()

pid_to_name = dict(zip(meta['product_id'], meta['name_clean']))
name_to_lvl2 = dict(zip(meta['name_clean'], meta['art_grp_lvl_2_name']))
name_to_lvl1 = dict(zip(meta['name_clean'], meta['art_grp_lvl_1_name']))
name_to_lvl0 = dict(zip(meta['name_clean'], meta['art_grp_lvl_0_name']))

ease_to_vae = {}
for ease_idx, pid in idx_to_product.items():
    if pid in pid_to_name:
        name = pid_to_name[pid]
        if name in item_to_idx_vae:
            ease_to_vae[ease_idx] = item_to_idx_vae[name]

print("Matched items:", len(ease_to_vae), "of", len(idx_to_product))

Matched items: 27212 of 27212


In [21]:
# MULTIVIEW + VAE FEATURES

vae_item_emb = model.item_dec.weight.detach().cpu().numpy()
vae_item_emb = vae_item_emb / (np.linalg.norm(vae_item_emb, axis=1, keepdims=True) + 1e-8)

def build_multiview_from_items(item_names):
    x_item = np.zeros(len(item_to_idx_vae), dtype=np.float32)
    x_lvl2 = np.zeros(len(lvl2_to_idx), dtype=np.float32)
    x_lvl1 = np.zeros(len(lvl1_to_idx), dtype=np.float32)
    x_lvl0 = np.zeros(len(lvl0_to_idx), dtype=np.float32)

    for name in item_names:
        if name in item_to_idx_vae:
            x_item[item_to_idx_vae[name]] = 1.0

        lvl2_name = name_to_lvl2.get(name)
        if lvl2_name in lvl2_to_idx:
            x_lvl2[lvl2_to_idx[lvl2_name]] = 1.0

        lvl1_name = name_to_lvl1.get(name)
        if lvl1_name in lvl1_to_idx:
            x_lvl1[lvl1_to_idx[lvl1_name]] = 1.0

        lvl0_name = name_to_lvl0.get(name)
        if lvl0_name in lvl0_to_idx:
            x_lvl0[lvl0_to_idx[lvl0_name]] = 1.0

    return x_item, x_lvl2, x_lvl1, x_lvl0


def vae_scores_in_ease_space(context):
    context_names = []
    for ease_idx in context:
        pid = idx_to_product[ease_idx]
        name = pid_to_name.get(pid)
        if name is not None:
            context_names.append(name)

    x = build_multiview_from_items(context_names)
    x = {
        'item': torch.FloatTensor(x[0]).unsqueeze(0).to(device),
        'lvl2': torch.FloatTensor(x[1]).unsqueeze(0).to(device),
        'lvl1': torch.FloatTensor(x[2]).unsqueeze(0).to(device),
        'lvl0': torch.FloatTensor(x[3]).unsqueeze(0).to(device)
    }

    with torch.no_grad():
        recon, _, _ = model(x, sample=False)
        raw_scores = recon['item'].cpu().numpy().flatten()

    scores = np.zeros(len(idx_to_product), dtype=np.float32)
    for ease_idx, vae_idx in ease_to_vae.items():
        scores[ease_idx] = raw_scores[vae_idx]

    return scores


def basket_emb_vae(context):
    vecs = []
    for ease_idx in context:
        if ease_idx in ease_to_vae:
            vecs.append(vae_item_emb[ease_to_vae[ease_idx]])
    if not vecs:
        return np.zeros(vae_item_emb.shape[1], dtype=np.float32)

    emb = np.mean(vecs, axis=0)
    emb = emb / (np.linalg.norm(emb) + 1e-8)
    return emb.astype(np.float32)

In [22]:
# HYBRID FEATURES

def build_features(ease_scores, vae_scores, candidates, context, ranks, b_emb):
    e = ease_scores[candidates].astype(np.float32)
    v = vae_scores[candidates].astype(np.float32)

    # z score
    e = (e - e.mean()) / (e.std() + 1e-8)
    v = (v - v.mean()) / (v.std() + 1e-8)

    feats = []

    for i, cand in enumerate(candidates):
        if cand in ease_to_vae:
            cand_emb = vae_item_emb[ease_to_vae[cand]]
            cos_vae = float(np.dot(b_emb, cand_emb))
        else:
            cos_vae = 0.0

        feats.append([
            e[i],    # 0: normalized ease
            v[i],    # 1: normalized vae score
            cos_vae,    # 2: basket-item cosine in VAE space
            e[i] * v[i],           
            e[i] * cos_vae,        
            v[i] * cos_vae,        
            float(len(context)),   
            float(ranks[i]),       
        ])

    return np.array(feats, dtype=np.float32)

In [23]:
# BUILD RANKING DATASET

def build_ranking_dataset(X_source, n_samples=5000, topk_candidates=200, seed=1211):
    rng = np.random.default_rng(seed)
    indices = rng.choice(X_source.shape[0], size=min(n_samples, X_source.shape[0]), replace=False)

    X_feat = []
    y = []
    group = []

    for i in tqdm(indices):
        row = X_source[i].toarray().flatten()
        items = np.where(row == 1)[0]

        if len(items) < 2:
            continue

        target = rng.choice(items)
        context = [x for x in items if x != target]

        if not context:
            continue

        ease_scores = np.zeros(B.shape[0], dtype=np.float32)
        for c in context:
            ease_scores += B[c]
        ease_scores[context] = -np.inf

        candidates = np.argpartition(-ease_scores, topk_candidates)[:topk_candidates]

        if target not in candidates:
            candidates = np.append(candidates[:-1], target)

        vae_scores = vae_scores_in_ease_space(context)
        b_emb = basket_emb_vae(context)

        cand_scores = ease_scores[candidates]
        order = np.argsort(-cand_scores)
        ranks = np.empty_like(order)
        ranks[order] = np.arange(len(candidates))

        X_local = build_features(ease_scores, vae_scores, candidates, context, ranks, b_emb)

        for j, cand in enumerate(candidates):
            X_feat.append(X_local[j])
            y.append(int(cand == target))

        group.append(len(candidates))

    return np.array(X_feat, dtype=np.float32), np.array(y, dtype=np.int32), group


X_rank, y_rank, group = build_ranking_dataset(
    X_train,
    n_samples=5000,
    topk_candidates=100,
    seed=1211
)

print("X_rank:", X_rank.shape)
print("y_rank:", y_rank.shape)
print("groups:", len(group))
print("positive rate:", y_rank.mean())

100%|██████████| 5000/5000 [01:00<00:00, 83.11it/s]


X_rank: (497600, 8)
y_rank: (497600,)
groups: 4976
positive rate: 0.01


In [26]:
print("TRAIN FINAL LOGISTIC RERANKER")

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_rank)

reranker = LogisticRegression(
    C=0.05,
    max_iter=300,
    class_weight="balanced",
    solver="lbfgs"
)

reranker.fit(X_scaled, y_rank);

TRAIN FINAL LOGISTIC RERANKER


In [27]:
def hybrid_scores(context, topk_candidates=200, alpha=0.5):
    
    ease_scores = np.zeros(B.shape[0], dtype=np.float32)
    for c in context:
        ease_scores += B[c]
    ease_scores[context] = -np.inf

    candidates = np.argpartition(-ease_scores, topk_candidates)[:topk_candidates]

    vae_scores = vae_scores_in_ease_space(context)
    b_emb = basket_emb_vae(context)

    cand_scores = ease_scores[candidates]
    order = np.argsort(-cand_scores)
    ranks = np.empty_like(order)
    ranks[order] = np.arange(len(candidates))

    X_feat = build_features(ease_scores, vae_scores, candidates, context, ranks, b_emb)
    X_feat = scaler.transform(X_feat)

    probs = reranker.predict_proba(X_feat)[:, 1]

    e = cand_scores
    e = (e - e.mean()) / (e.std() + 1e-8)

    p = probs
    p = (p - p.mean()) / (p.std() + 1e-8)

    final_scores = e + alpha * p

    return candidates, final_scores

In [28]:
def recall_hybrid_model(X_test, hybrid_fn, n_samples=5000, k=10, seed=42):
    
    rng = np.random.default_rng(seed)
    indices = rng.choice(X_test.shape[0], size=n_samples, replace=False)

    hits = 0
    total = 0

    for i in tqdm(indices):
        row = X_test[i].toarray().flatten()
        items = np.where(row == 1)[0]

        if len(items) < 2:
            continue

        target = rng.choice(items)
        context = [x for x in items if x != target]

        candidates, scores = hybrid_fn(context)

        top_k = candidates[np.argsort(-scores)[:k]]

        if target in top_k:
            hits += 1
        
        total += 1

    return hits / total if total > 0 else 0.0

In [ ]:
alphas = [0.05, 0.1, 0.3, 0.5, 0.7, 1, 10, 50]

best_recall, best_alpha = 0, 0

for a in alphas:
    
    def hybrid_scores_local(context):
        return hybrid_scores(context, alpha=a)
    
    score = recall_hybrid_model(X_test, hybrid_fn=hybrid_scores_local)
        
    if score >= best_recall:
        best_recall = score
        best_alpha = a

In [ ]:
print(f'Лучший Recall: {best_recall:.6f} при alpha: {best_alpha}')

Лучший Recall: 0.140157 при alpha: 0.3


In [ ]:
print("EASE Recall@10:   0.1249")
print("VAE Recall@10:    0.0862")
print(f"Hybrid Recall@10: {best_recall:.4f}")

EASE Recall@10:   0.1249
VAE Recall@10:    0.0862
Hybrid Recall@10: 0.1402


В результате применения двухэтапной гибридной архитектуры удалось повысить качество рекомендаций с 0.1249 до 0.1402 по метрике Recall@10.

**Обработка новых товаров (SKU)**  

В реальных условиях ассортимент товаров постоянно обновляется, поэтому необходимо учитывать сценарий добавления новых SKU без переобучения модели.  

В предложенной системе используется инкрементальный подход. Для модели EASE веса для нового товара вычисляются на основе его совместных появлений с другими товарами в корзинах, без изменения ранее обученных параметров, т.е. мы идём в базу и берём (50–100 чеков) чеков с новым SKU, у нового SKU должно быть хотя бы 20–30 разных соседних товаров по корзинам  

Для модели VAE латентное представление нового товара формируется через инференс по корзинам, в которых он встречается. Полученные представления позволяют встроить новый товар в существующее пространство признаков  

При этом модуль ранжирования (reranking) не требует переобучения, так как он оперирует универсальными признаками (оценки EASE, VAE и их комбинации), которые могут быть вычислены для любого нового товара  

Такой подход позволяет обеспечить стабильность системы и избежать смещения уже обученных представлений  

Результатом работы модели являются не только рекомендации, но и товарные признаки, которые могут использоваться в других бизнес-задачах. Для каждого SKU формируется набор признаков двух типов:  
- поведенческие признаки на основе модели EASE, отражающие силу связей товара с другими товарами в корзинах;  
- латентные признаки на основе модели VAE, представляющие товар в виде компактного embedding-вектора.  

Эти признаки могут быть использованы в downstream-задачах, включая модели аплифта, классификации, прогнозирования спроса и персонализации.

In [31]:
# ПРИЗНАКИ ДЛЯ SKU

def get_sku_features(product_id, topn_ease=10):
    if product_id not in product_to_idx:
        return None

    ease_idx = product_to_idx[product_id]

    #EASE
    ease_weights = B[ease_idx]
    top_idx = np.argsort(-ease_weights)[:topn_ease]
    top_related = [(idx_to_product[i], float(ease_weights[i])) for i in top_idx]

    # VAE
    if ease_idx in ease_to_vae:
        vae_idx = ease_to_vae[ease_idx]
        vae_embedding = vae_item_emb[vae_idx]
    else:
        vae_embedding = None

    return {
        "product_id": product_id,
        "ease_top_related": top_related,
        "vae_embedding": vae_embedding
    }

In [35]:
# пример: признаки для нескольких SKU из чека

sample_basket = df_final.groupby('cheque_id')['product_id'].apply(list).iloc[0]
sample_items = sample_basket[:3]

for pid in sample_items:
    feat = get_sku_features(pid, topn_ease=5)
    print()
    print("SKU:", feat["product_id"])
    print("Top EASE relations:")
    for rel_pid, w in feat["ease_top_related"]:
        print(f"  {rel_pid}: {w:.4f}")
    print("VAE embedding head:")
    print(feat["vae_embedding"][:10] if feat["vae_embedding"] is not None else None)


SKU: 161
Top EASE relations:
  10051: 0.0256
  9208: 0.0224
  10694: 0.0211
  5396: 0.0201
  10927: 0.0188
VAE embedding head:
[-0.08221839 -0.04410303  0.01131748  0.00054982 -0.17397475  0.16019492
  0.12535119  0.06193257  0.01204268  0.02367474]

SKU: 5538
Top EASE relations:
  2150: 0.0384
  2097: 0.0192
  11841: 0.0160
  7750: 0.0158
  682: 0.0154
VAE embedding head:
[-0.1897934   0.0321171   0.02385608 -0.02241921 -0.04027959 -0.0192392
  0.01450914  0.06427246 -0.01118544 -0.09607391]

SKU: 12965
Top EASE relations:
  6732: 0.0128
  17539: 0.0102
  161: 0.0081
  26583: 0.0080
  5538: 0.0080
VAE embedding head:
[-0.20004451 -0.16478261  0.1137806   0.00334631 -0.13784781  0.03038075
  0.10004074  0.0860519  -0.12418089 -0.0224403 ]


In [ ]:
# а это для более человеческого вида
all_lists_basket = df_final.groupby('cheque_id')['product_id'].apply(list).iloc[random.randint(1,50)]
sample_items = all_lists_basket[:1]

id_to_name = dict(zip(df_final['product_id'], df_final['name_clean']))

for pid in sample_items:
    feat = get_sku_features(pid, topn_ease=5)
    
    print("SKU:", id_to_name.get(feat["product_id"], feat["product_id"]))
    
    print("\nTop связанные товары (EASE):")
    for rel_pid, w in feat["ease_top_related"]:
        print(f"  {id_to_name.get(rel_pid, rel_pid)} → {w:.4f}")

SKU: винстон сс серые сигареты мрц178 джти  10 500

Top связанные товары (EASE):
  винстон сс серые сигареты мрц183 джти  10 500 → 0.0334
  бананы 1кг → 0.0214
  картофель 1кг → 0.0187
  алексеевское молоко сгущ с сах гост 8 5  650г д п  кмкк  12 → 0.0154
  цыпленок бройлер охл  1 кат  поли уп 1кг  13 → 0.0149


In [22]:
feat["ease_top_related"]

[(6732, 0.012814116664230824),
 (17539, 0.010195049457252026),
 (161, 0.00807187519967556),
 (26583, 0.007978049106895924),
 (5538, 0.007977960631251335)]

**Интерпретация результатов моделей**  

В рамках работы были построены две модели, каждая из которых формирует разный тип представления товара:  
**VAE: латентное представление товара**  
Для каждого SKU модель VAE строит вектор фиксированной размерности (embedding):  
SKU → [x1, x2, ..., x64]  
Пример:  
VAE embedding (первые 10 компонент):  
[-0.0822, -0.0441, 0.0113, ..., 0.0237]  
Что это значит: это компактное представление товара, оно отражает общий контекст потребления, близкие товары в этом пространстве имеют похожие паттерны покупок  
это аналог “смысла товара” в корзинах  

**EASE: поведенческий профиль товара**  
В отличие от VAE, модель EASE не строит embedding, а формирует:  
SKU → вектор связей с другими товарами  
То есть для каждого товара мы получаем:  
насколько он связан с каждым другим SKU  
Пример (сырой вывод)  
SKU: 161  
Top EASE relations:  
  10051: 0.0256  
  9208: 0.0224  
  ...  
Интерпретация (читаемый вид)  
SKU: цветная капуста замороженная  
Связанные товары:  
  брокколи замороженная → 0.0244  
  перец красный         → 0.0207  
  фасоль стручковая     → 0.0176  
  зеленый горошек       → 0.0126  
Что это значит: это уже не абстрактные числа, а поведенческий профиль товара   
То есть: с чем товар покупают, какие паттерны потребления он формирует, в какие корзины он попадает    

Сравнение моделей    

| Модель | Что выдаёт       | Смысл                   |  
| ------ | ---------------- | ----------------------- |  
| VAE    | embedding (64-d) | латентное представление |  
| EASE   | вектор связей    | поведенческий профиль   |  

Как из этого сделать полезные фичи  
Сырые выходы моделей не используются напрямую.  
Они преобразуются в признаки для downstream задач.  

1. Сила связей (EASE)  
sum(B[i])  
показывает:  
насколько товар “встроен” в корзины  
насколько он часто участвует в совместных покупках  

2. Максимальная связь  
max(B[i])  
показывает:  
есть ли у товара сильная “якорная” связь  
например: пиво → чипсы  

3. Связь с категорией  
sum(B[i][товары категории])  
позволяет понять:  
склонность товара к категории  
например:  
cola → snacks  
cola → алкоголь  

4. Топ-связи  
top-5 связанных товаров  
используется для:  
объяснимости  
аналитики  
построения дополнительных фич  

5. VAE embedding  
64 числовых признака  
используется как:  
универсальное представление товара  
вход в модели:  
uplift  
классификации  
персонализации  
  
Что мы отдаём коллегам  
Для каждого SKU формируется итоговый feature vector:  
SKU → [VAE embedding (64)] + [EASE агрегаты] + [EASE топ-связи / категории]  
Пример итогового набора признаков  
cola:  
 - vae_emb_1 ... vae_emb_64  
 - ease_strength = 0.42  
 - ease_max = 0.12  
 - snack_affinity = 0.25  
 - alcohol_affinity = 0.18  
 - top_related = [chips, nuts, beer]  
Практическая ценность   

Эти признаки могут использоваться в:

*uplift-моделях:*  
- вероятность отклика на товар  
- реакция на промо  

*рекомендательных системах:*  
- ранжирование кандидатов  
- персонализация  

*аналитике:*  
- поиск похожих товаров  
- анализ корзин  
- сегментация ассортимента  

*моделях прогнозирования*:  
- спрос  
- cross-sell  

**Финальный вывод**  
В результате работы была построена система, которая формирует два типа представлений товара:  
• латентное (VAE)  
• поведенческое (EASE)   
Их комбинация позволяет не только улучшить рекомендации, но и использовать модель как универсальный генератор признаков для различных бизнес-задач  